In [1]:
!python -V

Python 3.9.23


In [2]:
import pandas as pd
import numpy as np
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error


In [4]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

2025/07/29 01:45:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/07/29 01:45:37 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1', creation_time=1753506252722, experiment_id='1', last_update_time=1753506252722, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [5]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [6]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [7]:
len(df_train), len(df_val)

(73908, 61921)

In [8]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [9]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [10]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [11]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)


rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("RMSE:", rmse)

RMSE: 7.758715209663881


In [12]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [13]:
with mlflow.start_run():

    mlflow.set_tag("developer", "Phan Dang Ha")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    # rmse = root_mean_squared_error(y_val, y_pred)
    # rmse = mean_squared_error(y_val, y_pred, squared=False)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin",artifact_path="models_pickle")

In [14]:
import xgboost as xgb
from xgboost import XGBRegressor


In [15]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [16]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [18]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=2000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=500
        )
        y_pred = booster.predict(valid)
        # rmse = root_mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [19]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': scope.int(hp.quniform('seed', 42, 62, 1))
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=5,
    trials=Trials()
)

  0%|          | 0/5 [00:00<?, ?trial/s, best loss=?]

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [01:46:57] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:7.10425                          
[1]	validation-rmse:6.66307                          
[2]	validation-rmse:6.58150                          
[3]	validation-rmse:6.55569                          
[4]	validation-rmse:6.54500                          
[5]	validation-rmse:6.52531                          
[6]	validation-rmse:6.52299                          
[7]	validation-rmse:6.51645                          
[8]	validation-rmse:6.51023                          
[9]	validation-rmse:6.50639                          
[10]	validation-rmse:6.50151                         
[11]	validation-rmse:6.49869                         
[12]	validation-rmse:6.49751                         
[13]	validation-rmse:6.49507                         
[14]	validation-rmse:6.49550                         
[15]	validation-rmse:6.49304                         
[16]	validation-rmse:6.48973                         
[17]	validation-rmse:6.48825                         
[18]	validation-rmse:6.48676

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [01:50:11] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.03607                                                   
[1]	validation-rmse:10.08788                                                   
[2]	validation-rmse:9.33049                                                    
[3]	validation-rmse:8.73009                                                    
[4]	validation-rmse:8.26033                                                    
[5]	validation-rmse:7.89370                                                    
[6]	validation-rmse:7.60900                                                    
[7]	validation-rmse:7.38899                                                    
[8]	validation-rmse:7.21768                                                    
[9]	validation-rmse:7.08488                                                    
[10]	validation-rmse:6.98231                                                   
[11]	validation-rmse:6.90114                                                   
[12]	validation-rmse:6.83771            

...

[0]	validation-rmse:11.05781                                                   
[1]	validation-rmse:10.11881                                                   
[2]	validation-rmse:9.36173                                                    
[3]	validation-rmse:8.75631                                                    
[4]	validation-rmse:8.27686                                                    
[5]	validation-rmse:7.89968                                                    
[6]	validation-rmse:7.60201                                                    
[7]	validation-rmse:7.37063                                                    
[8]	validation-rmse:7.19040                                                    
[9]	validation-rmse:7.04998                                                    
[10]	validation-rmse:6.94056                                                   
[11]	validation-rmse:6.85557                                                   
[12]	validation-rmse:6.78766            

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [01:59:49] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.74026                                                   
[1]	validation-rmse:11.30350                                                   
[2]	validation-rmse:10.90080                                                   
[3]	validation-rmse:10.52984                                                   
[4]	validation-rmse:10.18844                                                   
[5]	validation-rmse:9.87498                                                    
[6]	validation-rmse:9.58686                                                    
[7]	validation-rmse:9.32313                                                    
[8]	validation-rmse:9.08194                                                    
[9]	validation-rmse:8.86158                                                    
[10]	validation-rmse:8.66026                                                   
[11]	validation-rmse:8.47671                                                   
[12]	validation-rmse:8.30948            

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [02:05:19] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.52247                                                   
[1]	validation-rmse:10.91008                                                   
[2]	validation-rmse:10.36866                                                   
[3]	validation-rmse:9.89099                                                    
[4]	validation-rmse:9.47145                                                    
[5]	validation-rmse:9.10429                                                    
[6]	validation-rmse:8.78337                                                    
[7]	validation-rmse:8.50358                                                    
[8]	validation-rmse:8.26036                                                    
[9]	validation-rmse:8.04948                                                    
[10]	validation-rmse:7.86732                                                   
[11]	validation-rmse:7.70938                                                   
[12]	validation-rmse:7.57293            

In [21]:
mlflow.xgboost.autolog(disable=True)

In [20]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)
    
    best_params = {
        'learning_rate':    0.07411882787027682,
        'max_depth':    52,
        'min_child_weight':    1.752932154225768,
        'reg_alpha':    0.34374319303872397,
        'reg_lambda':   0.36034741418636235,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 42
        
    }
    
    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50    
    )

    y_pred = booster.predict(valid)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)
    
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")


[0]	validation-rmse:11.60932
[1]	validation-rmse:11.06352
[2]	validation-rmse:10.57127
[3]	validation-rmse:10.12688
[4]	validation-rmse:9.72761
[5]	validation-rmse:9.37046
[6]	validation-rmse:9.05010
[7]	validation-rmse:8.76468
[8]	validation-rmse:8.50901
[9]	validation-rmse:8.28249
[10]	validation-rmse:8.07875
[11]	validation-rmse:7.90209
[12]	validation-rmse:7.74392
[13]	validation-rmse:7.60497
[14]	validation-rmse:7.48226
[15]	validation-rmse:7.37192
[16]	validation-rmse:7.27374
[17]	validation-rmse:7.18807
[18]	validation-rmse:7.11140
[19]	validation-rmse:7.04431
[20]	validation-rmse:6.98507
[21]	validation-rmse:6.93092
[22]	validation-rmse:6.88433
[23]	validation-rmse:6.84138
[24]	validation-rmse:6.80457
[25]	validation-rmse:6.77083
[26]	validation-rmse:6.74078
[27]	validation-rmse:6.71476
[28]	validation-rmse:6.69030
[29]	validation-rmse:6.66919
[30]	validation-rmse:6.65032
[31]	validation-rmse:6.63294
[32]	validation-rmse:6.61671
[33]	validation-rmse:6.60295
[34]	validation-rmse

2025/07/29 02:24:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [02:24:14] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/07/29 02:24:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [22]:
from mlflow.models.signature import infer_signature

input_example = X_val[:5]  # hoặc dùng pd.DataFrame với 5 hàng đầu tiên
signature = infer_signature(X_val, y_pred[:5])


In [23]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)
    
    best_params = {
        'learning_rate':    0.07411882787027682,
        'max_depth':    52,
        'min_child_weight':    1.752932154225768,
        'reg_alpha':    0.34374319303872397,
        'reg_lambda':   0.36034741418636235,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 42
        
    }
    
    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50    
    )

    y_pred = booster.predict(valid)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)
    
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(
    booster,
    artifact_path="models_mlflow",
    signature=signature,
    input_example=input_example
    )



[0]	validation-rmse:11.60932
[1]	validation-rmse:11.06352
[2]	validation-rmse:10.57127
[3]	validation-rmse:10.12688
[4]	validation-rmse:9.72761
[5]	validation-rmse:9.37046
[6]	validation-rmse:9.05010
[7]	validation-rmse:8.76468
[8]	validation-rmse:8.50901
[9]	validation-rmse:8.28249
[10]	validation-rmse:8.07875
[11]	validation-rmse:7.90209
[12]	validation-rmse:7.74392
[13]	validation-rmse:7.60497
[14]	validation-rmse:7.48226
[15]	validation-rmse:7.37192
[16]	validation-rmse:7.27374
[17]	validation-rmse:7.18807
[18]	validation-rmse:7.11140
[19]	validation-rmse:7.04431
[20]	validation-rmse:6.98507
[21]	validation-rmse:6.93092
[22]	validation-rmse:6.88433
[23]	validation-rmse:6.84138
[24]	validation-rmse:6.80457
[25]	validation-rmse:6.77083
[26]	validation-rmse:6.74078
[27]	validation-rmse:6.71476
[28]	validation-rmse:6.69030
[29]	validation-rmse:6.66919
[30]	validation-rmse:6.65032
[31]	validation-rmse:6.63294
[32]	validation-rmse:6.61671
[33]	validation-rmse:6.60295
[34]	validation-rmse

2025/07/29 02:35:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [02:35:15] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
